# Práctica: Prompt Engineering
## Parte 1 — Exploración de Técnicas

**Objetivo:** Experimentar con distintas técnicas de prompt engineering usando Azure AI Foundry con el modelo `gpt-4o`, entendiendo su propósito y efectividad en diferentes contextos.

---

### Estructura del notebook
1. Configuración del entorno
2. Técnicas de Prompt Engineering (6 técnicas)
3. Aplicación en casos de uso reales (2 casos)
4. Conclusiones

---
## 1. Configuración del entorno

In [1]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

AZURE_ENDPOINT    = os.getenv("AZURE_ENDPOINT")
AZURE_API_KEY     = os.getenv("AZURE_API_KEY")
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")
DEPLOYMENT_NAME   = os.getenv("DEPLOYMENT_NAME")

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

def chat(messages, temperature=0.7, max_tokens=800):
    """Helper para llamar al modelo y devolver el texto de respuesta."""
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

print("✅ Cliente Azure OpenAI configurado correctamente")

✅ Cliente Azure OpenAI configurado correctamente


---
## 2. Técnicas de Prompt Engineering

A continuación se exploran **6 técnicas** fundamentales, cada una con un ejemplo ejecutable y su análisis.

---
### Técnica 1: Zero-Shot Prompting

**¿Qué es?**  
Consiste en pedirle al modelo que realice una tarea directamente, sin darle ningún ejemplo previo de cómo hacerla. Solo se describe la tarea con claridad. El modelo usa únicamente su conocimiento previo para responder.

**¿Cuándo usarla?**  
Cuando la tarea es sencilla o el modelo ya tiene suficiente contexto por su entrenamiento. Es el punto de partida antes de recurrir a técnicas más complejas.

In [10]:
messages = [
    {
        "role": "user",
        "content": "Clasifica el sentimiento de la siguiente reseña como Positivo, Negativo o Neutro:\n\n'El producto llegó en perfecto estado y antes de lo esperado. Lo recomendaría sin dudarlo.'"
    }
]

respuesta = chat(messages)
print(respuesta)

messages = [
    {
        "role": "user",
        "content": "Clasifica el sentimiento de la siguiente reseña como Positivo, Negativo o Neutro:\n\n'Menuda mierda de servicio, no lo recomiendo.'"
    }
]

respuesta = chat(messages)
print(respuesta)

**Positivo**
Negativo


**Análisis del resultado:**  
> **Va bien :D**.  Zero-shot funciona bien en tareas de clasificación directa porque el modelo ya entrenó con millones de ejemplos de este tipo. Si la tarea fuera más específica o ambigua, probablemente necesitaríamos few-shot.

---
### Técnica 2: Few-Shot Prompting

**¿Qué es?**  
Se le proporcionan al modelo entre 2 y 5 ejemplos del formato input→output esperado antes de hacerle la pregunta real. De esta forma, el modelo "aprende" el patrón en contexto sin necesidad de reentrenamiento.

**¿Cuándo usarla?**  
Cuando el formato de salida es muy específico, cuando el dominio es técnico o especializado, o cuando zero-shot produce resultados inconsistentes.

In [14]:
messages = [
    {
        "role": "user",
        "content": """Extrae el producto y el precio de cada frase. Formato de salida: Producto: X | Precio: Y€

Ejemplo 1:
Frase: "Me compré unos auriculares Sony por 89 euros y estoy encantado."
Salida: Producto: auriculares Sony | Precio: 89€

Ejemplo 2:
Frase: "El portátil Lenovo que pedí costó 749 euros y llegó ayer."
Salida: Producto: portátil Lenovo | Precio: 749€

Ahora extrae de la siguiente frase:
Frase: "Acabo de pagar 35 pavos por un dragón mexicano que hace tacos muy buenos."
Salida:"""
    }
]

respuesta = chat(messages)
print(respuesta)

Producto: dragón mexicano | Precio: 35€


**Análisis del resultado:**  
> **También funciona :D**. Few-shot es especialmente poderoso para tareas de extracción de información donde el formato importa mucho. El modelo aprende el patrón "Producto: X | Precio: Y€" de los ejemplos y lo replica en la nueva frase, incluso con cosas que no tienen mucho sentido como la compra de un dragón. La clave es el formato que le das en los ejemplos.

---
### Técnica 3: Chain of Thought (CoT)

**¿Qué es?**  
Se le pide al modelo que explique su razonamiento paso a paso antes de dar la respuesta final. Esto mejora drásticamente el rendimiento en problemas que requieren múltiples pasos de inferencia, matemáticas o lógica.

**¿Cuándo usarla?**  
Problemas de lógica, matemáticas, toma de decisiones complejas, o cualquier tarea donde el camino al resultado importa tanto como el resultado mismo.

In [15]:
# Primero SIN Chain of Thought
messages_sin_cot = [
    {
        "role": "user",
        "content": "Una tienda tiene 240 artículos. Vende el 30% el lunes y el 25% de lo que queda el martes. ¿Cuántos artículos quedan?"
    }
]

# Luego CON Chain of Thought
messages_con_cot = [
    {
        "role": "user",
        "content": "Una tienda tiene 240 artículos. Vende el 30% el lunes y el 25% de lo que queda el martes. ¿Cuántos artículos quedan?\n\nRazona paso a paso antes de dar la respuesta final."
    }
]

print("=== SIN Chain of Thought ===")
print(chat(messages_sin_cot))
print("\n=== CON Chain of Thought ===")
print(chat(messages_con_cot))

=== SIN Chain of Thought ===
Vamos a resolver el problema paso a paso.

### Paso 1: Calcular el 30% de los artículos vendidos el lunes
La tienda tiene **240 artículos** al principio. El lunes vende el **30%** de ellos:

\[
30\% \text{ de } 240 = \frac{30}{100} \times 240 = 0.3 \times 240 = 72
\]

El lunes se venden **72 artículos**.

### Paso 2: Restar los artículos vendidos el lunes
Después del lunes, los artículos restantes son:

\[
240 - 72 = 168
\]

Quedan **168 artículos** después del lunes.

### Paso 3: Calcular el 25% de los artículos vendidos el martes
El martes, la tienda vende el **25%** de los artículos que quedaron (168):

\[
25\% \text{ de } 168 = \frac{25}{100} \times 168 = 0.25 \times 168 = 42
\]

El martes se venden **42 artículos**.

### Paso 4: Restar los artículos vendidos el martes
Después del martes, los artículos restantes son:

\[
168 - 42 = 126
\]

### Respuesta final:
Quedan **126 artículos** después de las ventas del lunes y martes.

=== CON Chain of Thought =

**Análisis del resultado:**  
> El modelo es suficientemente capaz como para aplicar CoT solo en tareas obvias. El valor real de la técnica se ve en dos situaciones: 1. Problemas más ambiguos o complejos, donde sin la instrucción explícita el modelo sí que se lanza a dar una respuesta directa (a veces incorrecta). 2. Cuando necesitas que el razonamiento sea visible y auditable, no solo el resultado

In [16]:
# Primero SIN Chain of Thought
messages_sin_cot = [
    {
        "role": "user",
        "content": "Una tienda tiene 240 artículos. Vende el 30% el lunes y el 25% de lo que queda el martes. ¿Cuántos quedan? Responde solo con el número."
    }
]

# Luego CON Chain of Thought
messages_con_cot = [
    {
        "role": "user",
        "content": "Una tienda tiene 240 artículos. Vende el 30% el lunes y el 25% de lo que queda el martes. ¿Cuántos artículos quedan?\n\nRazona paso a paso antes de dar la respuesta final."
    }
]

print("=== SIN Chain of Thought ===")
print(chat(messages_sin_cot))
print("\n=== CON Chain of Thought ===")
print(chat(messages_con_cot))

=== SIN Chain of Thought ===
126

=== CON Chain of Thought ===
Para resolver este problema, procederemos paso a paso:

### Paso 1: Calcular cuántos artículos se vendieron el lunes
La tienda tiene inicialmente **240 artículos**. El lunes vende el **30%** de ellos. Para calcular el 30% de 240, usamos la fórmula de porcentaje:

\[
30\% \text{ de } 240 = \frac{30}{100} \times 240 = 0.3 \times 240 = 72
\]

Por lo tanto, el lunes se venden **72 artículos**.

### Paso 2: Calcular cuántos artículos quedan después del lunes
Si se vendieron 72 artículos el lunes, restamos esta cantidad del total inicial de 240:

\[
240 - 72 = 168
\]

Por lo tanto, después del lunes quedan **168 artículos**.

### Paso 3: Calcular cuántos artículos se vendieron el martes
El martes se vende el **25% de lo que queda** después del lunes, es decir, el 25% de 168. Para calcular el 25% de 168, usamos la fórmula de porcentaje:

\[
25\% \text{ de } 168 = \frac{25}{100} \times 168 = 0.25 \times 168 = 42
\]

Por lo tanto, e

**Análisis del resultado:**  
> Para forzar al modelo a saltarse el razonamiento en la versión sin CoT, y que quede más clara la diferencia cambiamos el prompt sin CoT por uno que invite menos al razonamiento, por ejemplo formulándolo como pregunta directa de respuesta corta.

> Así si que podemos ver más fácilmente la diferencia.

---
### Técnica 4: Role Prompting (Asignación de Rol)

**¿Qué es?**  
Se le asigna al modelo un personaje, experto o perspectiva específica desde el mensaje de sistema o al inicio del prompt. Esto condiciona el tono, vocabulario y enfoque de todas las respuestas.

**¿Cuándo usarla?**  
Cuando necesitas respuestas especializadas en un dominio concreto, cuando el tono importa (formal, técnico, pedagógico), o cuando quieres simular un experto específico.

In [17]:
# Sin rol
messages_sin_rol = [
    {"role": "user", "content": "Explícame qué es una API REST."}
]

# Con rol de profesor para principiantes
messages_con_rol = [
    {
        "role": "system",
        "content": "Eres un profesor de programación experto en explicar conceptos técnicos a personas sin conocimientos previos. Usas analogías del mundo cotidiano, evitas el jargon técnico innecesario y terminas siempre con un ejemplo práctico muy sencillo."
    },
    {"role": "user", "content": "Explícame qué es una API REST."}
]

print("=== SIN ROL ===")
print(chat(messages_sin_rol))
print("\n=== CON ROL DE PROFESOR ===")
print(chat(messages_con_rol))

=== SIN ROL ===
Claro, aquí tienes una explicación clara y sencilla:

Una **API REST** (o **RESTful API**) es una interfaz que permite que diferentes sistemas, aplicaciones o servicios puedan comunicarse entre sí a través de internet utilizando el protocolo HTTP. REST (por sus siglas en inglés, **Representational State Transfer**) es un estilo de arquitectura que define un conjunto de principios para diseñar este tipo de interfaces.

En términos simples, una API REST es como un "puente" que permite que dos aplicaciones intercambien datos de forma estructurada y predecible.

---

### Características clave de una API REST:
1. **Basada en recursos:**
   - En REST, todo se trata de **recursos**. Un recurso puede ser cualquier entidad que se quiera manejar, como usuarios, productos, pedidos, etc.
   - Cada recurso se identifica con una **URL única** (por ejemplo, `https://miapi.com/usuarios` para un recurso de usuarios).

2. **Operaciones CRUD a través de métodos HTTP:**
   - REST utiliza l

**Análisis del resultado:**  
> **Aquí también se ve fácil la diferencia**. El rol condiciona no solo el contenido sino la forma de comunicar. En este caso el rol de profesor hace que la respuesta del LLM tenga muchos más ejemplos para explicarte de forma sencilla y que cualquiera lo pueda entender. Esta técnica es muy útil en aplicaciones donde el modelo actúa como asistente especializado (soporte técnico, tutor, asesor legal, etc.).

---
### Técnica 5: Output Formatting (Formato estructurado de salida)

**¿Qué es?**  
Se especifica explícitamente en el prompt el formato exacto que debe tener la respuesta: JSON, Markdown, tabla, lista numerada, campos específicos, etc. Esto convierte las respuestas del modelo en datos procesables directamente por código.

**¿Cuándo usarla?**  
Cuando la salida del modelo va a ser consumida por otro sistema, cuando necesitas parsear la respuesta, o cuando la consistencia del formato es crítica.

In [18]:
import json

messages = [
    {
        "role": "system",
        "content": "Responde SIEMPRE en formato JSON válido, sin texto adicional antes ni después."
    },
    {
        "role": "user",
        "content": """Analiza la siguiente descripción de producto y extrae la información en este JSON exacto:
{
  "nombre": "",
  "categoria": "",
  "precio_eur": 0,
  "caracteristicas": [],
  "sentimiento_reseña": ""
}

Descripción: 'La cafetera Philips HD7546 es perfecta para el hogar. Tiene capacidad para 15 tazas, temporizador programable y mantiene el café caliente. Cuesta 45 euros. Los usuarios la adoran por su facilidad de uso.'"""
    }
]

respuesta_raw = chat(messages, temperature=0.1)  # temperatura baja para consistencia
print("Respuesta raw del modelo:")
print(respuesta_raw)

# Intentar parsear como JSON para verificar que es válido
try:
    datos = json.loads(respuesta_raw)
    print("\n✅ JSON parseado correctamente:")
    print(json.dumps(datos, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"\n❌ Error al parsear JSON: {e}")

Respuesta raw del modelo:
{
  "nombre": "Philips HD7546",
  "categoria": "Cafetera",
  "precio_eur": 45,
  "caracteristicas": [
    "Capacidad para 15 tazas",
    "Temporizador programable",
    "Mantiene el café caliente"
  ],
  "sentimiento_reseña": "positivo"
}

✅ JSON parseado correctamente:
{
  "nombre": "Philips HD7546",
  "categoria": "Cafetera",
  "precio_eur": 45,
  "caracteristicas": [
    "Capacidad para 15 tazas",
    "Temporizador programable",
    "Mantiene el café caliente"
  ],
  "sentimiento_reseña": "positivo"
}


**Análisis del resultado:**  
> **De locos**. Con temperatura baja y formato explícito el modelo produce JSON parseable directamente. Esta es la técnica más usada en producción cuando el modelo es parte de un pipeline de datos. La combinación de system prompt + temperatura baja + schema explícito maximiza la consistencia.

---
### Técnica 6: Iterative Refinement (Refinamiento iterativo)

**¿Qué es?**  
En lugar de intentar obtener el resultado perfecto en un solo prompt, se genera una primera versión y luego se pide al modelo que la mejore con instrucciones concretas: cambiar el tono, reducir la longitud, añadir detalle en un punto específico, etc. Cada iteración acerca la respuesta al objetivo deseado.

**¿Cuándo usarla?**  
Cuando el resultado necesita ajustes finos que son difíciles de especificar todos de golpe: redacción de textos, resúmenes con formato concreto, respuestas que deben cumplir múltiples requisitos a la vez.

In [19]:
# Texto de partida: un resumen que queremos ir refinando
TEXTO_ORIGINAL = """
La inteligencia artificial está cambiando muchas industrias. Las empresas la usan para automatizar
tareas, analizar datos y mejorar la atención al cliente. También hay preocupaciones sobre el empleo
y la privacidad. En el futuro, la IA será aún más importante.
"""

# ── Iteración 1: primera versión del resumen ──────────────────────────────────
messages_v1 = [
    {
        "role": "user",
        "content": f"Resume el siguiente texto en 3 viñetas:\n{TEXTO_ORIGINAL}"
    }
]

v1 = chat(messages_v1)
print("=== VERSIÓN 1 (resumen inicial) ===")
print(v1)

# ── Iteración 2: pedir tono más formal y eliminar redundancia ─────────────────
messages_v2 = [
    *messages_v1,
    {"role": "assistant", "content": v1},
    {
        "role": "user",
        "content": "Bien. Ahora reescríbelo con un tono más formal y profesional, y reduce cada viñeta a un máximo de 20 palabras."
    }
]

v2 = chat(messages_v2)
print("\n=== VERSIÓN 2 (más formal, más conciso) ===")
print(v2)

# ── Iteración 3: añadir una viñeta sobre implicaciones futuras ────────────────
messages_v3 = [
    *messages_v2,
    {"role": "assistant", "content": v2},
    {
        "role": "user",
        "content": "Perfecto. Añade una cuarta viñeta que mencione específicamente el impacto en el mercado laboral, manteniendo el mismo tono y extensión que las anteriores."
    }
]

v3 = chat(messages_v3)
print("\n=== VERSIÓN 3 (versión final con 4 viñetas) ===")
print(v3)

=== VERSIÓN 1 (resumen inicial) ===
- La inteligencia artificial transforma industrias mediante la automatización, el análisis de datos y la mejora de la atención al cliente.  
- Surgen preocupaciones sobre su impacto en el empleo y la privacidad.  
- Su relevancia aumentará aún más en el futuro.  

=== VERSIÓN 2 (más formal, más conciso) ===
- La inteligencia artificial impulsa cambios al optimizar procesos, automatizar tareas y mejorar la experiencia del cliente en diversas industrias.  
- Existen inquietudes significativas respecto a su impacto en la privacidad y la estabilidad del mercado laboral.  
- La importancia de la inteligencia artificial seguirá creciendo exponencialmente en los próximos años.  

=== VERSIÓN 3 (versión final con 4 viñetas) ===
- La inteligencia artificial impulsa cambios al optimizar procesos, automatizar tareas y mejorar la experiencia del cliente en diversas industrias.  
- Existen inquietudes significativas respecto a su impacto en la privacidad y la est

**Análisis del resultado:**  
> **Va bien también**. Cada iteración afina un aspecto concreto sin rehacer todo desde cero. La clave de Iterative Refinement es que cada instrucción de mejora es específica (tono, longitud, un punto concreto a añadir), no genérica. Cuanto más precisa es la instrucción de refinamiento, más predecible y útil es el resultado.

---
## 3. Aplicación en Casos de Uso Reales

En esta sección se aplican las técnicas más apropiadas a dos escenarios reales con justificación de la elección.

### Caso 1: Extracción de datos estructurados de contratos

**Contexto:** Una empresa necesita procesar automáticamente correos de solicitud de hipoteca y extraer los datos clave en formato estructurado para ingresarlos en un CRM.

**Técnicas aplicadas:**
- **Few-Shot**: Para enseñarle al modelo el formato de extracción específico del negocio.
- **Output Formatting (JSON)**: Para que la salida sea directamente consumible por el sistema.
- **Role Prompting**: Para que el modelo actúe como analista financiero y use terminología correcta.

**Justificación:** La combinación de few-shot + JSON es ideal aquí porque el formato de extracción es muy específico y la salida va a un sistema externo. El rol añade precisión en el dominio financiero.

In [20]:
CORREO_SOLICITUD = """
Buenos días,

Me llamo Carlos Fernández, tengo 38 años y estoy interesado en solicitar una hipoteca 
para la compra de un piso en Madrid. El precio del inmueble es de 320.000 euros y 
necesitaría financiar el 80%, es decir, unos 256.000 euros a 25 años.

Mi situación laboral es contrato indefinido con una antigüedad de 7 años en la misma empresa. 
Cobro 2.800 euros netos al mes. No tengo otras deudas actualmente.

¿Podrían decirme qué opciones tendrían disponibles?

Saludos,
Carlos Fernández
Tel: 612 345 678
"""

messages = [
    {
        "role": "system",
        "content": "Eres un analista de préstamos hipotecarios. Tu tarea es extraer datos de solicitudes de clientes en formato JSON estructurado. Responde SOLO con el JSON, sin texto adicional."
    },
    {
        "role": "user",
        "content": f"""Extrae los datos del siguiente correo siguiendo el esquema:
{{
  "nombre_cliente": "",
  "edad": null,
  "telefono": "",
  "valor_inmueble_eur": null,
  "importe_solicitado_eur": null,
  "plazo_años": null,
  "ltv_porcentaje": null,
  "ingresos_netos_mensuales_eur": null,
  "tipo_contrato": "",
  "antigüedad_laboral_años": null,
  "otras_deudas": null
}}

Correo:
{CORREO_SOLICITUD}"""
    }
]

respuesta = chat(messages, temperature=0.0)
print(respuesta)

# Parsear y mostrar estructurado
try:
    datos = json.loads(respuesta)
    print("\n✅ Datos extraídos correctamente para el CRM:")
    for k, v in datos.items():
        print(f"  {k}: {v}")
except:
    print("Revisar formato de respuesta")

```json
{
  "nombre_cliente": "Carlos Fernández",
  "edad": 38,
  "telefono": "612 345 678",
  "valor_inmueble_eur": 320000,
  "importe_solicitado_eur": 256000,
  "plazo_años": 25,
  "ltv_porcentaje": 80,
  "ingresos_netos_mensuales_eur": 2800,
  "tipo_contrato": "indefinido",
  "antigüedad_laboral_años": 7,
  "otras_deudas": false
}
```
Revisar formato de respuesta


**Análisis:**  
> Este caso demuestra el valor real de Output Formatting en producción. La combinación de system prompt como analista + JSON estricto + temperatura 0 produce datos directamente insertables en una base de datos sin procesamiento adicional.

---
### Caso 2: Redacción de email comercial para captación de clientes

**Contexto:** Una empresa necesita redactar emails de prospección para enviar a potenciales clientes. El primer borrador raramente cumple todos los requisitos a la vez (tono, longitud, llamada a la acción), por lo que se refina iterativamente.

**Técnicas aplicadas:**
- **Role Prompting**: Para que el modelo adopte el perfil de un copywriter especializado en ventas B2B.
- **Iterative Refinement**: Para ajustar el borrador inicial en varias pasadas (tono → longitud → llamada a la acción).
- **Output Formatting**: Para obtener la versión final en un formato listo para copiar y enviar.

**Justificación:** Un email comercial tiene muchas restricciones simultáneas (breve, persuasivo, personalizado, con CTA clara) que son difíciles de especificar todas en un solo prompt. Iterative Refinement permite ir añadiendo requisitos uno a uno hasta llegar al resultado deseado.

In [21]:
CONTEXTO_EMPRESA = """
Empresa: Iatenea
Producto: Sistema de automatización de seguimiento de leads para intermediarios financieros
Propuesta de valor: Reduce el tiempo de seguimiento manual un 80% y aumenta la tasa de conversión
Destinatario: Director de una correduría hipotecaria de tamaño medio
"""

# ── Iteración 1: borrador inicial con rol de copywriter ───────────────────────
messages_email_v1 = [
    {
        "role": "system",
        "content": "Eres un copywriter especializado en ventas B2B para empresas de tecnología. Escribes emails de prospección directos, sin relleno y centrados en el valor para el cliente."
    },
    {
        "role": "user",
        "content": f"Escribe un email de prospección frío para captar un cliente potencial con este contexto:\n{CONTEXTO_EMPRESA}"
    }
]

email_v1 = chat(messages_email_v1, temperature=0.7)
print("=== BORRADOR 1 (versión inicial) ===")
print(email_v1)

# ── Iteración 2: acortar y eliminar cualquier tono agresivo ───────────────────
messages_email_v2 = [
    *messages_email_v1,
    {"role": "assistant", "content": email_v1},
    {
        "role": "user",
        "content": "Bien como base. Ahora acórtalo: el cuerpo no debe superar 5 líneas. Elimina cualquier frase que suene a venta agresiva y usa un tono consultivo, como si propusieras una conversación, no una venta."
    }
]

email_v2 = chat(messages_email_v2, temperature=0.7)
print("\n=== BORRADOR 2 (más corto, tono consultivo) ===")
print(email_v2)

# ── Iteración 3: añadir una CTA concreta y asunto optimizado ──────────────────
messages_email_v3 = [
    *messages_email_v2,
    {"role": "assistant", "content": email_v2},
    {
        "role": "user",
        "content": "Perfecto. Dos últimos ajustes: (1) la llamada a la acción final debe proponer una reunión de 20 minutos con fecha abierta, (2) escribe también un asunto para el email de máximo 8 palabras que genere curiosidad sin ser clickbait. Devuelve el email completo listo para enviar con el asunto en la primera línea."
    }
]

email_v3 = chat(messages_email_v3, temperature=0.7)
print("\n=== VERSIÓN FINAL (con asunto y CTA) ===")
print(email_v3)

=== BORRADOR 1 (versión inicial) ===
**Asunto:** Multiplica tu tasa de conversión y ahorra un 80% del tiempo en el seguimiento de leads  

Hola [Nombre del destinatario],  

Gestionar múltiples leads mientras mantienes un seguimiento personalizado es un desafío, especialmente en una correduría hipotecaria donde cada oportunidad cuenta.  

En Iatenea hemos desarrollado un sistema de automatización de seguimiento de leads diseñado específicamente para intermediarios financieros como tú. Nuestra plataforma reduce el tiempo dedicado al seguimiento manual en un 80%, mientras ayuda a incrementar la tasa de conversión al mantener a los leads siempre comprometidos.  

Imagina a tu equipo dedicando menos tiempo a tareas repetitivas y más a cerrar negocios. Empresas como [nombre de un cliente similar, si lo hay] ya están viendo resultados claros: más cierres, menos fricción.  

¿Te interesaría una breve conversación para explorar cómo podríamos ayudarte a optimizar tu proceso de conversión?  

U

**Análisis:**  
> La combinación de CoT + Self-Consistency produce documentación más completa que un prompt directo. El modelo analiza primero antes de documentar, y luego se autocorrige. Esto es especialmente útil en pipelines de CI/CD donde la documentación se genera automáticamente.

---
## 4. Conclusiones

Todas las técnicas son útiles dependiendo del uso que necesites. Conocerlas al final permite ser capaz de funcionar de forma más eficiente para las distintas situaciones que te encuentres en el futuro, así que diría que todas me parecen más o menos útiles.

Lo importante desde mi punto de vista es entenderlas para saber cuándo usarlas eficientemente. 
